# 🧩 Customer Segmentation Analysis
**Pipeline:** Upload → Evaluate → Clean → K-Means Clustering → Visualize

Run each cell in order from top to bottom.

In [1]:
# Cell 1: Install required libraries
!pip install pandas numpy scikit-learn matplotlib seaborn plotly ipywidgets openpyxl -q

In [2]:
# Cell 2: Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
import io
import os

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_samples, silhouette_score
from sklearn.impute import SimpleImputer

from ipywidgets import FileUpload, Output, Button, VBox, HBox, Label
from IPython.display import display, clear_output, HTML

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print('✅ All libraries imported successfully.')

✅ All libraries imported successfully.


In [5]:
# Cell 3: Upload your data file (CSV, Excel, or JSON)
upload = FileUpload(accept='.csv,.xlsx,.xls,.json', multiple=False)
upload_output = Output()

display(HTML('<h3>📂 Step 1: Upload Your Customer Data File</h3>'))
display(HTML('<p>Supported formats: <b>CSV, Excel (.xlsx/.xls), JSON</b></p>'))
display(upload)
display(upload_output)

FileUpload(value=(), accept='.csv,.xlsx,.xls,.json', description='Upload')

Output()

In [7]:
# Cell 4: Load and inspect the uploaded file
def load_data(upload_widget):
    if not upload_widget.value:
        print('⚠️  No file uploaded yet. Please upload a file in Cell 3.')
        return None

    uploaded_file = list(upload_widget.value.values())[0]
    filename = uploaded_file['metadata']['name']
    content = uploaded_file['content']
    file_bytes = io.BytesIO(bytes(content))

    ext = os.path.splitext(filename)[1].lower()
    if ext == '.csv':
        df = pd.read_csv(file_bytes)
    elif ext in ['.xlsx', '.xls']:
        df = pd.read_excel(file_bytes)
    elif ext == '.json':
        df = pd.read_json(file_bytes)
    else:
        print(f'❌ Unsupported file format: {ext}')
        return None

    print(f'✅ File loaded: {filename}')
    print(f'📊 Shape: {df.shape[0]} rows × {df.shape[1]} columns\n')
    return df

df_raw = load_data(upload)

if df_raw is not None:
    display(HTML('<h3>🔍 Raw Data Preview</h3>'))
    display(df_raw.head(10))

AttributeError: 'tuple' object has no attribute 'values'

In [ ]:
# Cell 5: Evaluate and profile the dataset
def evaluate_data(df):
    print('=' * 60)
    print('         📋 DATA EVALUATION REPORT')
    print('=' * 60)

    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    date_cols = df.select_dtypes(include=['datetime']).columns.tolist()

    print(f'\n📌 Total Columns    : {len(df.columns)}')
    print(f'   Numeric columns  : {len(num_cols)} → {num_cols}')
    print(f'   Categorical cols : {len(cat_cols)} → {cat_cols}')
    print(f'   Date columns     : {len(date_cols)} → {date_cols}')
    print(f'\n📌 Total Rows       : {len(df)}')
    print(f'   Duplicate rows   : {df.duplicated().sum()}')

    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
    missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

    if not missing_df.empty:
        print(f'\n⚠️  Columns with Missing Values:')
        display(missing_df)
    else:
        print('\n✅ No missing values detected.')

    print('\n📊 Descriptive Statistics (Numeric Columns):')
    display(df[num_cols].describe().round(2))

    high_card = [c for c in cat_cols if df[c].nunique() > 50]
    if high_card:
        print(f'\n⚠️  High-cardinality categorical columns (>50 unique): {high_card}')

    return num_cols, cat_cols

if df_raw is not None:
    num_cols, cat_cols = evaluate_data(df_raw)

In [ ]:
# Cell 6: Automated data cleaning
def clean_data(df, num_cols, cat_cols):
    print('=' * 60)
    print('         🧹 DATA CLEANING')
    print('=' * 60)
    df_clean = df.copy()

    dupes = df_clean.duplicated().sum()
    df_clean = df_clean.drop_duplicates()
    print(f'\n✅ Removed {dupes} duplicate rows.')

    thresh = 0.6
    high_missing = [c for c in df_clean.columns if df_clean[c].isnull().mean() > thresh]
    df_clean = df_clean.drop(columns=high_missing)
    if high_missing:
        print(f'✅ Dropped high-missing columns (>{int(thresh*100)}%): {high_missing}')

    num_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = df_clean.select_dtypes(include=['object', 'category']).columns.tolist()

    if num_cols:
        imputer_num = SimpleImputer(strategy='median')
        df_clean[num_cols] = imputer_num.fit_transform(df_clean[num_cols])
        print('✅ Imputed missing numeric values with median.')

    if cat_cols:
        imputer_cat = SimpleImputer(strategy='most_frequent')
        df_clean[cat_cols] = imputer_cat.fit_transform(df_clean[cat_cols])
        print('✅ Imputed missing categorical values with mode.')

    outlier_counts = {}
    for col in num_cols:
        Q1 = df_clean[col].quantile(0.01)
        Q3 = df_clean[col].quantile(0.99)
        before = len(df_clean)
        df_clean = df_clean[(df_clean[col] >= Q1) & (df_clean[col] <= Q3)]
        removed = before - len(df_clean)
        if removed > 0:
            outlier_counts[col] = removed
    if outlier_counts:
        print(f'✅ Removed extreme outliers (1st-99th percentile): {outlier_counts}')

    print(f'\n📊 Cleaned Dataset: {df_clean.shape[0]} rows × {df_clean.shape[1]} columns')
    print('\n✅ Cleaning complete.')
    return df_clean, num_cols, cat_cols

if df_raw is not None:
    df_clean, num_cols, cat_cols = clean_data(df_raw, num_cols, cat_cols)
    display(HTML('<h3>✨ Cleaned Data Preview</h3>'))
    display(df_clean.head())

In [ ]:
# Cell 7: Prepare features for clustering
def prepare_features(df, num_cols):
    print('=' * 60)
    print('         ⚙️  FEATURE PREPARATION')
    print('=' * 60)

    if not num_cols:
        print('❌ No numeric columns found for clustering.')
        return None, None, None

    features = df[num_cols].copy()
    print(f'\n📌 Features selected for clustering ({len(num_cols)}):\n   {num_cols}')

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(features)
    print(f'\n✅ Features standardized (mean=0, std=1)')

    return X_scaled, features, scaler

if df_raw is not None:
    X_scaled, features, scaler = prepare_features(df_clean, num_cols)

In [ ]:
# Cell 8: Determine the optimal number of clusters
def find_optimal_k(X_scaled, k_min=2, k_max=10):
    print('=' * 60)
    print('         🔍 FINDING OPTIMAL NUMBER OF CLUSTERS')
    print('=' * 60)

    inertias = []
    silhouette_scores = []
    k_range = range(k_min, k_max + 1)

    for k in k_range:
        km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
        labels = km.fit_predict(X_scaled)
        inertias.append(km.inertia_)
        sil = silhouette_score(X_scaled, labels)
        silhouette_scores.append(sil)
        print(f'   k={k:2d} | Inertia: {km.inertia_:,.1f} | Silhouette: {sil:.4f}')

    best_k = list(k_range)[np.argmax(silhouette_scores)]
    print(f'\n🏆 Best k by Silhouette Score: {best_k}')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
    axes[0].axvline(x=best_k, color='red', linestyle='--', label=f'Best k={best_k}')
    axes[0].set_title('Elbow Method — Inertia', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Number of Clusters (k)')
    axes[0].set_ylabel('Inertia (Within-Cluster SSE)')
    axes[0].legend()

    axes[1].plot(k_range, silhouette_scores, 'go-', linewidth=2, markersize=8)
    axes[1].axvline(x=best_k, color='red', linestyle='--', label=f'Best k={best_k}')
    axes[1].set_title('Silhouette Score by k', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Number of Clusters (k)')
    axes[1].set_ylabel('Silhouette Score')
    axes[1].legend()

    plt.suptitle('Optimal K Selection', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('optimal_k.png', bbox_inches='tight', dpi=150)
    plt.show()
    print("✅ Plot saved as 'optimal_k.png'")

    return best_k, inertias, silhouette_scores

if df_raw is not None:
    best_k, inertias, sil_scores = find_optimal_k(X_scaled)

In [ ]:
# Cell 9: Run K-Means with the optimal k
# You can override best_k here if desired, e.g. K = 4
K = best_k

def run_kmeans(X_scaled, df_clean, k):
    print('=' * 60)
    print(f'         🤖 RUNNING K-MEANS (k={k})')
    print('=' * 60)

    km = KMeans(n_clusters=k, init='k-means++', n_init=20, random_state=42)
    labels = km.fit_predict(X_scaled)

    df_result = df_clean.copy()
    df_result['Cluster'] = labels
    df_result['Cluster'] = df_result['Cluster'].astype(str)

    sil = silhouette_score(X_scaled, labels)
    print(f'\n✅ K-Means complete.')
    print(f'   Silhouette Score : {sil:.4f}')
    print(f'   Inertia          : {km.inertia_:,.1f}')
    print(f'\n📊 Cluster Distribution:')
    display(df_result['Cluster'].value_counts().sort_index().rename('Count').to_frame())

    return df_result, km, labels

if df_raw is not None:
    df_result, km_model, labels = run_kmeans(X_scaled, df_clean, K)

In [ ]:
# Cell 10: Silhouette plot for cluster quality
def plot_silhouette(X_scaled, labels, k):
    fig, ax = plt.subplots(figsize=(10, 6))
    sil_vals = silhouette_samples(X_scaled, labels)
    avg_sil = silhouette_score(X_scaled, labels)

    y_lower = 10
    colors = cm.tab10(np.linspace(0, 1, k))

    for i in range(k):
        ith_sil = np.sort(sil_vals[labels == i])
        size_i = ith_sil.shape[0]
        y_upper = y_lower + size_i
        ax.fill_betweenx(np.arange(y_lower, y_upper), 0, ith_sil,
                         facecolor=colors[i], edgecolor=colors[i], alpha=0.7, label=f'Cluster {i}')
        ax.text(-0.05, y_lower + 0.5 * size_i, str(i), fontsize=10)
        y_lower = y_upper + 10

    ax.axvline(x=avg_sil, color='red', linestyle='--', label=f'Avg silhouette = {avg_sil:.3f}')
    ax.set_title(f'Silhouette Plot (k={k})', fontsize=14, fontweight='bold')
    ax.set_xlabel('Silhouette Coefficient')
    ax.set_ylabel('Cluster')
    ax.legend(loc='upper right')
    plt.tight_layout()
    plt.savefig('silhouette_plot.png', bbox_inches='tight', dpi=150)
    plt.show()
    print("✅ Plot saved as 'silhouette_plot.png'")

if df_raw is not None:
    plot_silhouette(X_scaled, labels, K)

In [ ]:
# Cell 11: PCA 2D scatter plot of clusters
def plot_pca_clusters(X_scaled, labels, k):
    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(X_scaled)
    ev = pca.explained_variance_ratio_

    pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
    pca_df['Cluster'] = labels.astype(str)

    fig = px.scatter(
        pca_df, x='PC1', y='PC2', color='Cluster',
        title=f'Customer Segments — PCA 2D Projection (k={k})<br>'
              f'<sup>PC1: {ev[0]*100:.1f}% variance | PC2: {ev[1]*100:.1f}% variance</sup>',
        color_discrete_sequence=px.colors.qualitative.Bold,
        labels={'PC1': f'PC1 ({ev[0]*100:.1f}% var)', 'PC2': f'PC2 ({ev[1]*100:.1f}% var)'},
        opacity=0.75, width=850, height=550
    )
    fig.update_traces(marker=dict(size=7))
    fig.update_layout(legend_title_text='Cluster')
    fig.write_html('pca_clusters.html')
    fig.show()
    print("✅ Interactive plot saved as 'pca_clusters.html'")

if df_raw is not None:
    plot_pca_clusters(X_scaled, labels, K)

In [ ]:
# Cell 12: Cluster profiles — mean feature values + heatmap + radar chart
def plot_cluster_profiles(df_result, num_cols):
    profile = df_result.groupby('Cluster')[num_cols].mean()
    profile_norm = (profile - profile.min()) / (profile.max() - profile.min() + 1e-9)

    display(HTML('<h3>📋 Cluster Mean Feature Profiles</h3>'))
    display(profile.round(2))

    fig, ax = plt.subplots(figsize=(max(10, len(num_cols)), max(4, len(profile))))
    sns.heatmap(profile_norm, annot=True, fmt='.2f', cmap='YlOrRd', linewidths=0.5,
                ax=ax, cbar_kws={'label': 'Normalized Value'})
    ax.set_title('Cluster Feature Profiles (Normalized)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Feature')
    ax.set_ylabel('Cluster')
    plt.tight_layout()
    plt.savefig('cluster_heatmap.png', bbox_inches='tight', dpi=150)
    plt.show()
    print("✅ Heatmap saved as 'cluster_heatmap.png'")

    categories = num_cols
    N = len(categories)
    if N >= 3:
        angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
        angles += angles[:1]

        fig_r, ax_r = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
        colors = plt.cm.tab10(np.linspace(0, 1, len(profile_norm)))

        for idx, (cluster, row) in enumerate(profile_norm.iterrows()):
            values = row.tolist() + [row.tolist()[0]]
            ax_r.plot(angles, values, 'o-', linewidth=2, color=colors[idx], label=f'Cluster {cluster}')
            ax_r.fill(angles, values, alpha=0.1, color=colors[idx])

        ax_r.set_xticks(angles[:-1])
        ax_r.set_xticklabels(categories, size=10)
        ax_r.set_title('Cluster Radar Chart', size=15, fontweight='bold', pad=20)
        ax_r.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
        plt.tight_layout()
        plt.savefig('cluster_radar.png', bbox_inches='tight', dpi=150)
        plt.show()
        print("✅ Radar chart saved as 'cluster_radar.png'")
    else:
        print('ℹ️  Not enough features for radar chart (need ≥ 3).')

    return profile

if df_raw is not None:
    cluster_profile = plot_cluster_profiles(df_result, num_cols)

In [ ]:
# Cell 13: Box plots — feature distributions per cluster
def plot_feature_distributions(df_result, num_cols):
    n = len(num_cols)
    cols_per_row = 3
    rows = (n + cols_per_row - 1) // cols_per_row
    fig, axes = plt.subplots(rows, cols_per_row, figsize=(18, 5 * rows))
    axes = axes.flatten()

    palette = sns.color_palette('tab10', n_colors=df_result['Cluster'].nunique())

    for i, col in enumerate(num_cols):
        sns.boxplot(data=df_result, x='Cluster', y=col, palette=palette, ax=axes[i])
        axes[i].set_title(f'{col} by Cluster', fontweight='bold')
        axes[i].set_xlabel('Cluster')

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('Feature Distributions Across Clusters', fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('feature_distributions.png', bbox_inches='tight', dpi=150)
    plt.show()
    print("✅ Distribution plot saved as 'feature_distributions.png'")

if df_raw is not None:
    plot_feature_distributions(df_result, num_cols)

In [ ]:
# Cell 14: Interactive 3D PCA cluster plot
def plot_3d_pca(X_scaled, labels, num_cols):
    if X_scaled.shape[1] < 3:
        print('ℹ️  Not enough features for 3D PCA (need ≥ 3).')
        return

    pca3 = PCA(n_components=3, random_state=42)
    X_3d = pca3.fit_transform(X_scaled)
    ev = pca3.explained_variance_ratio_

    df_3d = pd.DataFrame(X_3d, columns=['PC1', 'PC2', 'PC3'])
    df_3d['Cluster'] = labels.astype(str)

    fig = px.scatter_3d(
        df_3d, x='PC1', y='PC2', z='PC3', color='Cluster',
        title=f'3D PCA Cluster View — {ev[0]*100:.1f}% + {ev[1]*100:.1f}% + {ev[2]*100:.1f}% variance',
        color_discrete_sequence=px.colors.qualitative.Bold,
        opacity=0.75, width=900, height=650
    )
    fig.update_traces(marker=dict(size=5))
    fig.write_html('pca_3d_clusters.html')
    fig.show()
    print("✅ 3D interactive plot saved as 'pca_3d_clusters.html'")

if df_raw is not None:
    plot_3d_pca(X_scaled, labels, num_cols)

In [ ]:
# Cell 15: Export results and print final summary
def export_and_summarize(df_result, cluster_profile, k):
    print('=' * 60)
    print('         📦 SUMMARY & EXPORT')
    print('=' * 60)

    out_file = 'customer_segments.csv'
    df_result.to_csv(out_file, index=False)
    print(f"\n✅ Segmented data exported → '{out_file}'")

    print(f'\n🎯 Final Cluster Summary (k={k}):\n')
    summary = df_result.groupby('Cluster').size().rename('Customer Count').to_frame()
    summary['% Share'] = (summary['Customer Count'] / len(df_result) * 100).round(1)
    display(summary)

    display(HTML('<h3>📌 Cluster Profile Means</h3>'))
    display(cluster_profile.round(3))

    print('\n📁 Files generated:')
    files = [
        'customer_segments.csv', 'optimal_k.png', 'silhouette_plot.png',
        'pca_clusters.html', 'cluster_heatmap.png', 'cluster_radar.png',
        'feature_distributions.png', 'pca_3d_clusters.html'
    ]
    for f in files:
        if os.path.exists(f):
            print(f'   ✅ {f}')

    display(HTML('<h2>🏁 Customer Segmentation Analysis Complete!</h2>'))

if df_raw is not None:
    export_and_summarize(df_result, cluster_profile, K)